In [ ]:
# Colab: GPU 런타임을 먼저 선택하고 이 셀 전체를 실행하세요.
DATA_DIR = '/content/drive/MyDrive/T2_data'
SEED = 42
VALID_FOLD = 0
BATCH_SIZE = 8
HEAD_EPOCHS = 3
FINE_TUNE_EPOCHS = 25
PATIENCE = 8
IMAGE_SIZE = (336, 224)  # (높이, 너비), 원본 비율을 거의 유지
NUM_WORKERS = 2

import importlib.util
import subprocess
import sys
required = {'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'pillow',
            'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'tqdm': 'tqdm'}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)

import hashlib
import json
import random
import shutil
import time
import zipfile
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import models, transforms
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from google.colab import drive, files

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU가 연결되지 않았습니다. Colab 런타임 유형을 GPU로 바꾸세요. '
        f'현재 torch={torch.__version__}, CUDA build={torch.version.cuda}'
    )
assert 0 <= VALID_FOLD < 5 and BATCH_SIZE >= 1
drive.mount('/content/drive')
print('GPU:', torch.cuda.get_device_name(0))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
device = torch.device('cuda')

# Drive 폴더를 로컬에 복사합니다. labels.csv가 한 단계 안쪽에 있어도 찾습니다.
source_dir = Path(DATA_DIR)
if not source_dir.is_dir():
    raise FileNotFoundError(f'DATA_DIR을 실제 Drive 폴더로 수정하세요: {source_dir}')
label_files = ([source_dir / 'labels.csv'] if (source_dir / 'labels.csv').is_file()
               else list(source_dir.rglob('labels.csv')))
if len(label_files) != 1:
    raise ValueError('DATA_DIR 아래 labels.csv가 정확히 하나 있어야 합니다.')
label_file = label_files[0]
local_root = Path('/content/task2_vgg_baseline_data')
data_root = local_root / 'T2_data'
manifest = pd.read_csv(label_file, dtype={'sample_id': str, 'patient_group': str})
assert {'sample_id', 'path', 'label', 'patient_group'}.issubset(manifest.columns)
for value in manifest.path:
    original_path = (label_file.parent / str(value)).resolve()
    if not original_path.is_relative_to(label_file.parent.resolve()) or not original_path.is_file():
        raise ValueError(f'이미지 경로를 확인하세요: {value}')
cached_labels = data_root / 'labels.csv'
ready = cached_labels.is_file() and cached_labels.read_bytes() == label_file.read_bytes()
ready = ready and all((data_root / str(p)).is_file() for p in manifest.path)
if not ready:
    print('이미지 폴더를 Colab 로컬 디스크로 복사합니다.', flush=True)
    shutil.copytree(label_file.parent, data_root, dirs_exist_ok=True)

labels = pd.read_csv(data_root / 'labels.csv', dtype={'sample_id': str, 'patient_group': str})
assert {'sample_id', 'path', 'label', 'patient_group'}.issubset(labels.columns)
assert not labels[['sample_id', 'path', 'label', 'patient_group']].isna().any().any()
assert labels.sample_id.is_unique
assert set(labels.label.unique()) == {0, 1}
for value in labels.path:
    image_path = (data_root / str(value)).resolve()
    assert image_path.is_relative_to(data_root.resolve()) and image_path.is_file(), value

# 모든 모델에서 재사용할 수 있는 환자별 fold 표를 한 번 저장합니다.
base_dir = source_dir.parent / 'task2_vgg_results'
base_dir.mkdir(parents=True, exist_ok=True)
fold_file = base_dir / 'patient_folds_seed42.csv'
hash_file = base_dir / 'patient_folds_seed42_metadata.json'
label_hash = hashlib.sha256(
    labels[['sample_id', 'path', 'label', 'patient_group']].to_csv(index=False).encode()
).hexdigest()

if fold_file.exists() or hash_file.exists():
    if not (fold_file.exists() and hash_file.exists()):
        raise ValueError('기존 환자 분리 파일 두 개가 모두 있는지 확인하세요.')
    metadata = json.loads(hash_file.read_text())
    assert metadata['label_hash'] == label_hash and metadata['seed'] == SEED, '기존 분리와 데이터/seed가 다릅니다.'
    old = pd.read_csv(fold_file, dtype={'sample_id': str, 'patient_group': str})
    assert set(old.sample_id) == set(labels.sample_id)
    labels = labels.merge(old[['sample_id', 'fold']], on='sample_id', validate='one_to_one')
else:
    labels['fold'] = -1
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (_, valid_idx) in enumerate(splitter.split(labels, labels.label, labels.patient_group)):
        labels.loc[valid_idx, 'fold'] = fold
    labels.to_csv(fold_file, index=False)
    hash_file.write_text(json.dumps(dict(label_hash=label_hash, seed=SEED), indent=2))

assert labels.groupby('patient_group')['fold'].nunique().eq(1).all()
assert labels.fold.between(0, 4).all()
train_df = labels[labels.fold.ne(VALID_FOLD)].reset_index(drop=True)
valid_df = labels[labels.fold.eq(VALID_FOLD)].reset_index(drop=True)
assert not set(train_df.patient_group) & set(valid_df.patient_group)
assert set(train_df.label.unique()) == set(valid_df.label.unique()) == {0, 1}
print('학습:', len(train_df), '장 /', train_df.patient_group.nunique(), '명')
print('검증:', len(valid_df), '장 /', valid_df.patient_group.nunique(), '명')
print('검증 환자:', sorted(valid_df.patient_group.unique()))
if SEED == 42 and VALID_FOLD == 0:
    assert set(valid_df.patient_group) == {'G06', 'G07', 'G09', 'G14', 'G17', 'G20'}, '기준 실험과 환자 분리가 다릅니다.'
    assert len(train_df) == 2349 and len(valid_df) == 599, '기준 실험과 이미지 수가 다릅니다.'


run_id = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S')
out = base_dir / f'vgg11_pretrained_fold{VALID_FOLD}_{run_id}'
out.mkdir(parents=True)
labels.to_csv(out / 'patient_folds.csv', index=False)
local_best = local_root / f'best_{run_id}.pt'

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=.5),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
valid_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class UltrasoundDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(data_root / row.path) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, int(row.label)

train_loader = DataLoader(
    UltrasoundDataset(train_df, train_transform), batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    generator=torch.Generator().manual_seed(SEED),
)
valid_loader = DataLoader(
    UltrasoundDataset(valid_df, valid_transform), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

# 사전학습 VGG의 마지막 출력층만 2개 클래스로 교체합니다.
model = models.vgg11(weights=models.VGG11_Weights.DEFAULT)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
model = model.to(device)
counts = np.bincount(train_df.label.to_numpy(int), minlength=2)
class_weights = torch.tensor(len(train_df) / (2 * counts), dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler = torch.amp.GradScaler('cuda')
config = dict(
    architecture='torchvision.vgg11', initialization='IMAGENET1K_V1', seed=SEED,
    valid_fold=VALID_FOLD, image_size=list(IMAGE_SIZE), normalize_mean=MEAN,
    normalize_std=STD, class_names=['absent', 'present'], positive_class=1,
    batch_size=BATCH_SIZE, head_epochs=HEAD_EPOCHS, fine_tune_epochs=FINE_TUNE_EPOCHS,
    experiment='baseline', brightness_jitter=0., contrast_jitter=0.,
    class_weights=class_weights.detach().cpu().tolist(), threshold=.5,
    train_groups=sorted(train_df.patient_group.unique().tolist()),
    valid_groups=sorted(valid_df.patient_group.unique().tolist()),
)
(out / 'config.json').write_text(json.dumps(config, indent=2))

def train_epoch(optimizer, description):
    model.train()
    numerator, denominator = 0., 0.
    for images, targets in tqdm(train_loader, desc=description, leave=False):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, targets)
        if not torch.isfinite(loss):
            raise ValueError('학습 손실이 유한하지 않습니다.')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.)
        scaler.step(optimizer)
        scaler.update()
        weight = float(class_weights[targets].sum().item())
        numerator += float(loss.item()) * weight
        denominator += weight
    return numerator / denominator

@torch.inference_mode()
def evaluate():
    model.eval()
    probability, target_list = [], []
    numerator, denominator = 0., 0.
    for images, targets in valid_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, targets)
        probability.extend(logits.float().softmax(dim=1)[:, 1].cpu().numpy())
        target_list.extend(targets.cpu().numpy())
        weight = float(class_weights[targets].sum().item())
        numerator += float(loss.item()) * weight
        denominator += weight
    y = np.asarray(target_list, dtype=int)
    p = np.asarray(probability, dtype=float)
    tn, fp, fn, tp = confusion_matrix(y, p >= .5, labels=[0, 1]).ravel()
    metrics = dict(valid_loss=numerator / denominator,
                   valid_ba=float(balanced_accuracy_score(y, p >= .5)),
                   valid_auroc=float(roc_auc_score(y, p)),
                   recall_absent=float(tn / (tn + fp)),
                   recall_present=float(tp / (tp + fn)))
    return metrics, y, p

history = []
best_key = (-1., -1.)
best_info = None
global_epoch = 0
for stage, epochs in [('head', HEAD_EPOCHS), ('finetune', FINE_TUNE_EPOCHS)]:
    if stage == 'head':
        for parameter in model.parameters():
            parameter.requires_grad = False
        for parameter in model.classifier[-1].parameters():
            parameter.requires_grad = True
        optimizer = torch.optim.AdamW(model.classifier[-1].parameters(), lr=1e-3, weight_decay=1e-4)
    else:
        checkpoint = torch.load(local_best, map_location='cpu', weights_only=True)
        model.load_state_dict(checkpoint['model_state'])
        del checkpoint
        for parameter in model.parameters():
            parameter.requires_grad = True
        optimizer = torch.optim.AdamW([
            {'params': model.features.parameters(), 'lr': 1e-5},
            {'params': model.classifier[:-1].parameters(), 'lr': 1e-5},
            {'params': model.classifier[-1].parameters(), 'lr': 1e-4},
        ], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=.5, patience=3, min_lr=1e-6,
    )
    wait = 0
    for stage_epoch in range(1, epochs + 1):
        global_epoch += 1
        start = time.perf_counter()
        train_loss = train_epoch(optimizer, f'{stage} {stage_epoch}/{epochs}')
        metrics, _, _ = evaluate()
        key = (metrics['valid_ba'], metrics['valid_auroc'])
        row = dict(epoch=global_epoch, stage=stage, stage_epoch=stage_epoch,
                   train_loss=train_loss, **metrics,
                   lr=float(optimizer.param_groups[0]['lr']),
                   seconds=round(time.perf_counter() - start, 1))
        history.append(row)
        pd.DataFrame(history).to_csv(out / 'history.csv', index=False)
        # Primary selection: BA at fixed threshold .5; AUROC only breaks ties.
        if key > best_key:
            best_key = key
            best_info = dict(row)
            torch.save(dict(model_state=model.state_dict(), config=config,
                            best_metrics=best_info, threshold=.5), local_best)
            wait = 0
        else:
            wait += 1
        scheduler.step(metrics['valid_ba'])
        # Local checkpoint is copied periodically to limit large Drive writes.
        if global_epoch == 1 or stage_epoch % 5 == 0:
            shutil.copy2(local_best, out / 'vgg.pt')
        print(f"{stage} {stage_epoch:02d}: loss={train_loss:.4f}, "
              f"Valid BA={metrics['valid_ba']:.4f}, "
              f"recall0={metrics['recall_absent']:.4f}, "
              f"recall1={metrics['recall_present']:.4f}, "
              f"best BA={best_key[0]:.4f}, {row['seconds']:.0f}s", flush=True)
        if stage == 'finetune' and wait >= PATIENCE:
            print('조기 종료: 검증 성능 개선이 멈췄습니다.', flush=True)
            break
    shutil.copy2(local_best, out / 'vgg.pt')

# Best epoch is fixed first. Threshold is then selected on the same Validation.
checkpoint = torch.load(local_best, map_location='cpu', weights_only=True)
model.load_state_dict(checkpoint['model_state'])
metrics, y, p = evaluate()
thresholds = np.linspace(.1, .9, 81)
threshold = float(max(thresholds, key=lambda t: (
    balanced_accuracy_score(y, p >= t), -abs(t - .5)
)))
tuned_ba = float(balanced_accuracy_score(y, p >= threshold))
checkpoint['threshold'] = threshold
checkpoint['config']['threshold'] = threshold
torch.save(checkpoint, local_best)
shutil.copy2(local_best, out / 'vgg.pt')
config['threshold'] = threshold
(out / 'config.json').write_text(json.dumps(config, indent=2))

predictions = valid_df[['sample_id', 'patient_group', 'label']].copy()
predictions['probability'] = p
predictions['prediction_05'] = (p >= .5).astype(int)
predictions['prediction_tuned'] = (p >= threshold).astype(int)
predictions.to_csv(out / 'valid_predictions.csv', index=False)
patient_rows = []
for group, part in predictions.groupby('patient_group'):
    patient_rows.append(dict(patient_group=group, images=len(part),
        positives=int(part.label.sum()),
        ba_05=float(balanced_accuracy_score(part.label, part.prediction_05)),
        ba_tuned=float(balanced_accuracy_score(part.label, part.prediction_tuned))))
pd.DataFrame(patient_rows).to_csv(out / 'patient_metrics.csv', index=False)
report = dict(
    architecture='VGG11', pretrained='ImageNet-1K', best_epoch=best_info['epoch'],
    best_stage=best_info['stage'], train_images=len(train_df), valid_images=len(valid_df),
    train_patients=train_df.patient_group.nunique(), valid_patients=valid_df.patient_group.nunique(),
    valid_ba_at_05=metrics['valid_ba'], valid_auroc=metrics['valid_auroc'],
    recall_absent_at_05=metrics['recall_absent'], recall_present_at_05=metrics['recall_present'],
    selected_threshold=threshold, valid_ba_tuned=tuned_ba,
    note='Single held-out patient fold used for epoch and threshold selection. These are Validation scores, not independent Test scores. No full-data refit or ensemble.',
)
(out / 'metrics.json').write_text(json.dumps(report, indent=2))
(out / 'requirements_actual.txt').write_text('\n'.join(
    f'{package}=={importlib.metadata.version(package)}' for package in
    ['torch', 'torchvision', 'numpy', 'pandas', 'pillow', 'scikit-learn', 'matplotlib']
) + '\n')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
h = pd.DataFrame(history)
axes[0].plot(h.epoch, h.train_loss, label='train (augmented)')
axes[0].plot(h.epoch, h.valid_loss, label='validation')
axes[0].set(xlabel='Epoch', ylabel='Weighted CE loss')
axes[0].legend()
axes[1].plot(h.epoch, h.valid_ba, label='validation BA @ .5')
axes[1].plot(h.epoch, h.valid_auroc, label='validation AUROC')
axes[1].axvline(best_info['epoch'], color='gray', linestyle='--')
axes[1].set(xlabel='Epoch', ylabel='Score')
axes[1].legend()
fig.tight_layout()
fig.savefig(out / 'learning_curves.png', dpi=150)
plt.show()

# Keep the large model on Drive; download only the small diagnostic report.
small_zip = out / 'vgg_report.zip'
with zipfile.ZipFile(small_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ['metrics.json', 'config.json', 'history.csv', 'patient_metrics.csv',
                 'valid_predictions.csv', 'patient_folds.csv', 'requirements_actual.txt', 'learning_curves.png']:
        archive.write(out / name, name)
print(json.dumps(report, indent=2))
print('가중치:', out / 'vgg.pt')
print('환자 분리표:', fold_file)
print('분석용 ZIP에는 가중치를 넣지 않았습니다. vgg.pt는 Drive에 보관됩니다.')
files.download(str(small_zip))
